# 00 — Complete MVTec AD dataset EDA

Research foundation for normal-only industrial anomaly detection in PyTorch.
This notebook is the chronological scientific narrative, together with the
[research protocol](../docs/research_protocol.md). No model is fitted here.

**Execution contract:** run top to bottom from the repository or notebooks folder.
Only this repository's data directory is searched. Missing data produces explicit
blocked sections, never placeholder benchmark statistics. Fixture tests validate
code independently and are not MVTec observations. Rendered outputs containing
dataset images stay local; commit a cleared notebook and compact numeric reports.

## 1. Introduction and questions

We compare normal-only image detection and defect localization across six
planned model families. First ask: is the complete dataset intact, what spatial
detail might resizing lose, and can fixed normal partitions support defensible
calibration? The primary input resolution is preregistered at 256×256, not selected
by test performance. Configuration and data provenance must remain traceable.

In [ ]:
from pathlib import Path
import sys, json, hashlib
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run from repository or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from anomaly_detection.data import (CATEGORIES, TEXTURES, project_data_root,
                                    discover, inventory, partitions)
from anomaly_detection.tracking import config_hash
config = json.loads((ROOT / "configs/eda.json").read_text())
print(json.dumps(config, indent=2))
print("Configuration SHA256:", config_hash(config))

The displayed configuration is the exact one used below. EDA performs no
hyperparameter selection, model fitting, threshold optimization or accuracy claim.

## 2. Origin and organization

The [official MVTec AD page](https://www.mvtec.com/research-teaching/datasets/mvtec-ad)
describes defect-free training images, normal and defective test images, and
pixel annotations for defects across 15 categories. Cite Bergmann et al. (CVPR
2019) and the extended IJCV paper. Data is CC BY-NC-SA 4.0. Follow the official
download form; see [acquisition instructions](../docs/dataset.md).
Expected hierarchy: `<category>/train/good`, `test/<defect or good>`, and
`ground_truth/<defect>/*_mask.png`. No other project is a data source.

In [ ]:
DATA = project_data_root(ROOT, config["dataset_root"])
OUT = ROOT / "outputs/eda"
OUT.mkdir(parents=True, exist_ok=True)
print("Project-local dataset root:", DATA.relative_to(ROOT))
print("Directory exists:", DATA.is_dir())

A directory alone does not establish completeness. Discovery and full decoding
below must succeed before reporting measured dataset statistics.

## 3. Environment and GPU inspection

Inspect actual imports, CUDA build/runtime availability and driver-visible
hardware. A driver's CUDA version is not proof that PyTorch has CUDA support.

In [ ]:
from environment import inspect
environment = inspect()
display(environment)
(OUT / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")

CPU-only PyTorch can perform this EDA. GPU model fitting requires a CUDA-enabled
build plus a successful device allocation; memory feasibility is not inferred
from an installed driver alone.

## 4. Discovery and integrity checks

Only the configured project-local root is allowed. Require all categories,
decode every image, verify mask shape/binary values and pairing, hash image files,
and reject duplicate content before normal partitioning. Partial data is blocked.
Large corrupt-image exceptions deliberately stop execution for investigation.

In [ ]:
found = discover(DATA)
missing = sorted(set(CATEGORIES) - set(found))
READY = not missing
frame = pd.DataFrame()
split_manifest = pd.DataFrame()
if READY:
    frame = inventory(DATA)
    split_manifest = partitions(frame, seed=config["seed"])
    frame.to_csv(OUT / "inventory.csv", index=False)
    split_manifest.to_csv(OUT / "partitions.csv", index=False)
    print("Decoded images:", len(frame))
else:
    print("BLOCKED: complete MVTec AD is unavailable at data/mvtec_ad.")
    print("Missing categories:", missing)
    print("No EDA statistics computed. Follow docs/dataset.md.")

When blocked, subsequent dataset sections explicitly skip computation. When
available, failures are evidence to investigate, not permission to drop records.
SHA256 detects byte-identical copies; it does not establish near-duplicate safety.

## 5. All category names

The expected coverage is fixed in advance. Texture/object membership describes
the benchmark taxonomy and is separate from locally measured counts.

In [ ]:
category_table = pd.DataFrame({"category": CATEGORIES,
    "kind": ["texture" if c in TEXTURES else "object" for c in CATEGORIES],
    "discovered": [c in found for c in CATEGORIES]})
display(category_table)

All 15 must be represented in the final comparison. An absent category is not
a category with zero images; it is unavailable data.

## 6. Image counts by category and split

Count decoded files, separating train normals, test normals and test defects.
These counts are measured only after complete integrity checks.

In [ ]:
if READY:
    counts = frame.groupby(["category", "split", "anomalous"]).size().unstack([1, 2], fill_value=0)
    display(counts.reindex(CATEGORIES))
    counts.to_csv(OUT / "counts.csv")
    counts.plot.bar(stacked=True, figsize=(12, 4), ylabel="Images")
    plt.title("Decoded image counts; test labels used descriptively only")
    plt.tight_layout(); plt.show()
else:
    print("SKIPPED: no verified image counts.")

The stacked count plot exposes category imbalance without inventing training
defects. Macro reporting will weight categories equally rather than by image count.

## 7. Normal and defective samples

Show one deterministic normal training image and one defective test image per
category. These are descriptive examples; do not use their appearance to tune
models. Test exposure must be disclosed in subsequent reports.

In [ ]:
if READY:
    fig, axes = plt.subplots(5, 6, figsize=(13, 11))
    for i, category in enumerate(CATEGORIES):
        subset = frame[frame.category == category]
        examples = [subset[subset.split == "train"].iloc[0],
                    subset[(subset.split == "test") & subset.anomalous].iloc[0]]
        for j, row in enumerate(examples):
            ax = axes[i // 3, (i % 3) * 2 + j]
            with Image.open(DATA / row.path) as image:
                ax.imshow(image.convert("RGB"))
            ax.set_title(f"{category} / {row.defect}", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("SKIPPED: no sample images.")

One example cannot characterize defect prevalence or severity. The compact
all-category view supports taxonomy inspection, not a performance conclusion.

## 8. Defect-type distributions

Enumerate actual test defect directories and counts. Defect frequency differs
from pixel area and does not justify supervised training under this protocol.

In [ ]:
if READY:
    defects = frame[(frame.split == "test") & frame.anomalous].groupby(["category", "defect"]).size().rename("images")
    display(defects.to_frame())
    defects.to_csv(OUT / "defect_types.csv")
else:
    print("SKIPPED: no verified defect-type counts.")

These annotations remain evaluation-only. Do not select configurations or
thresholds to favor frequent or visually prominent test defect types.

## 9. Dimensions, channels and formats

Measure native geometry and image encodings before transforming them. Original
channels are recorded even though the proposed shared transform converts to RGB.

In [ ]:
if READY:
    display(frame.groupby("category")[["width", "height", "channels"]].agg(["min", "max"]))
    display(frame.groupby(["mode", "format"]).size().rename("images").to_frame())
else:
    print("SKIPPED: dimensions/channels/formats unavailable.")

RGB conversion makes model input consistent but does not recover information
absent from grayscale images. Native sizes determine the amount of downsampling.

## 10. Mask availability

Every defective test image must have a nonempty, binary, native-resolution mask.
Normal images have no annotation file; zero masks are only their semantic normal
evaluation target, not generated defect labels.

In [ ]:
if READY:
    masks = frame.groupby(["category", "anomalous"]).agg(images=("path", "size"), masks=("mask", "count"))
    display(masks)
    assert frame.loc[frame.anomalous, "mask"].notna().all()
    assert frame.loc[~frame.anomalous, "mask"].isna().all()
else:
    print("SKIPPED: no verified image/mask pairs.")

Mask corruption, orphans, and missing annotations are fatal integrity errors.
Official masks never enter fitting, early stopping or threshold optimization.

## 11. Image and mask comparisons

Overlay one defective example per category at native geometry. Red indicates
official annotated pixels; the image itself is not transformed before overlay.

In [ ]:
if READY:
    fig, axes = plt.subplots(3, 5, figsize=(13, 8))
    for ax, category in zip(axes.flat, CATEGORIES):
        row = frame[(frame.category == category) & frame.anomalous].iloc[0]
        with Image.open(DATA / row.path) as image:
            ax.imshow(image.convert("RGB"))
        with Image.open(DATA / row["mask"]) as mask:
            foreground = np.asarray(mask) > 0
        ax.imshow(np.ma.masked_where(~foreground, foreground), cmap="Reds", alpha=.55, vmin=0, vmax=1)
        ax.set_title(category); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("SKIPPED: no annotated examples.")

Overlays can reveal alignment errors that file pairing alone misses. Figures
contain licensed dataset content and remain in local executed outputs.

## 12. Pixel intensities and image statistics

Summarize per-image RGB [0,1] intensity mean and population standard deviation
using official training normals only. These descriptive statistics are not fitted
normalization parameters; later normalization may use fit normals only.

In [ ]:
if READY:
    train = frame[frame.split == "train"]
    display(train.groupby("category")[["mean", "std"]].agg(["mean", "min", "max"]))
    train.boxplot(column="mean", by="category", figsize=(12, 4), rot=60)
    plt.ylabel("Per-image mean intensity [0,1]"); plt.suptitle("")
    plt.title("Training-normal brightness variation"); plt.tight_layout(); plt.show()
else:
    print("SKIPPED: no observed intensity statistics.")

Image means hide local contrast and defects. They can flag decode or exposure
issues but cannot demonstrate anomaly separability.

## 13. Textures versus objects

Five texture categories have repeated surface structure; ten object categories
include shape and component arrangement. Compare measured native geometry and
counts by this predefined grouping without treating categories as replicates.

In [ ]:
if READY:
    grouped = frame.assign(kind=frame.category.map(lambda c: "texture" if c in TEXTURES else "object"))
    display(grouped.groupby("kind").agg(images=("path", "size"), categories=("category", "nunique"),
        median_width=("width", "median"), median_height=("height", "median")))
else:
    print("Known taxonomy: 5 texture, 10 object categories; observed summaries unavailable.")

Texture/object subgroup performance will be supplementary. The primary report
still includes each category and the all-category unweighted macro mean.

## 14. Input resolution and resizing

Quantify scale factors for the preregistered 256×256 starting resolution. Whole
image resizing can distort aspect ratios and remove tiny defects. These geometry
calculations do not select a resolution by test-mask performance.

In [ ]:
if READY:
    geometry = frame[frame.split == "train"].copy()
    geometry["width_scale"] = config["resize_candidate"][0] / geometry.width
    geometry["height_scale"] = config["resize_candidate"][1] / geometry.height
    display(geometry.groupby("category")[["width_scale", "height_scale"]].agg(["min", "max"]))
else:
    print("Proposed 256x256 only; native scale factors cannot be measured.")

Image transforms use bilinear antialiasing; masks use nearest neighbor. Evaluate
continuous maps restored to original image size against original masks. Any
model-specific exception is recorded before final test evaluation.

## 15. Fit, development, calibration and test partitions

Freeze seed-2026 normal-only partitions: floor(15%) development, floor(15%)
calibration, remainder fitting per category. All official test files stay test.
The manifest stores relative paths and image hashes and is retained locally.

In [ ]:
if READY:
    display(split_manifest.groupby(["category", "partition"]).size().unstack(fill_value=0))
    manifest_sha = hashlib.sha256((OUT / "partitions.csv").read_bytes()).hexdigest()
    print("Partition manifest SHA256:", manifest_sha)
else:
    print("SKIPPED: no real partition assignments created.")

Fit learns representations/statistics; development guides declared normal or
synthetic proxy choices; calibration fixes thresholds after selection. No fitting
uses official test masks, labels, or held-out calibration images.

## 16. Leakage audits

Audit path uniqueness, content hashes, partition coverage and train/test
membership. Exact hashes cannot rule out perceptual duplicates or acquisition
groups. Inspect suspicious cases before fitting, documenting any resolution.

In [ ]:
if READY:
    assert len(split_manifest) == len(frame)
    assert split_manifest.path.is_unique and frame.sha256.is_unique
    merged = frame.merge(split_manifest[["path", "partition"]], on="path", validate="one_to_one")
    assert (merged.loc[merged.split == "test", "partition"] == "test").all()
    assert (merged.loc[merged.split == "train", "partition"] != "test").all()
    assert not merged.loc[merged.partition != "test", "anomalous"].any()
    print("PASS: exact-file and partition isolation checks. Near-duplicate review remains separate.")
else:
    print("BLOCKED: leakage audit requires official files.")

Viewing test examples here is disclosed label exposure. Future reports may
claim no test-label tuning only if that remains true; never claim unseen test
labels after this notebook has shown them.

## 17. Baseline evaluation definitions

Higher scores indicate anomalies. Image AUROC ranks positives above negatives;
AP is stepwise average precision. At the frozen normal-calibration 95th percentile
(higher order statistic), alarm iff score > threshold. Report recall, FPR,
precision, F1 and [[TN,FP],[FN,TP]]. Pixel AUROC/AP pool pixels within category;
AU-PRO averages 8-connected region overlap over background FPR ≤0.30 and divides
area by 0.30. Report per-category and 15-category macro results, image-bootstrap
intervals and seed variation. No real model scores exist yet.

In [ ]:
from anomaly_detection.metrics import binary_metrics, calibrate_normal
# Hand-checkable synthetic metric smoke check only; not MVTec performance.
fixture = binary_metrics([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8], threshold=.5)
assert fixture["auroc"] == .75
assert fixture["confusion_matrix"] == [[2, 0], [1, 1]]
print("Synthetic metric fixture passed; no benchmark result computed.")

Calibration normals contain no real positives, so they cannot estimate real
defect recall or optimize F1. The full metric, uncertainty and runtime definitions
are frozen in the research protocol before any training.

## 18. Initial architectural hypotheses

These are preregistered hypotheses, not findings:

| Family | Hypothesis and concern |
|---|---|
| Basic CNN | A declared one-class or synthetic proxy objective provides an inexpensive reference; proxy defects may not transfer to real defects. |
| Convolutional autoencoder | Normal reconstruction exposes local defects; an expressive decoder may reconstruct anomalies too well. |
| PatchCore | Pretrained patch neighbors may provide strong localization; memory bank size and search latency can dominate. |
| SuperSimpleNet | Synthetic feature perturbations may improve discrimination; proxy design can induce a domain gap. |
| PaDiM | Spatial Gaussian patch statistics may work for aligned objects; covariance memory and alignment sensitivity can limit use. |
| EfficientAD | Teacher/student and reconstruction disagreement may offer fast detection; training cost and weak small defects need examination. |

Each later notebook must show the exact architecture, objective, data transforms,
fit loop or fitting algorithm, configuration and evaluation steps. External
pretrained weights require provenance and a registered experiment need.

## 19. Hardware feasibility

Inspect actual hardware before proposing batch sizes. An 8 GB laptop GPU
suggests batch-1/low-batch experiments and memory-bounded feature extraction;
only measurements establish training feasibility. No models are downloaded.

In [ ]:
print("Driver-visible hardware:", environment.get("nvidia_smi", "unavailable"))
print("PyTorch CUDA available:", environment.get("cuda_available", False))
if not environment.get("cuda_available", False):
    print("BLOCKER for GPU training: CUDA-enabled PyTorch environment not verified.")
print("No fit, inference latency, or model peak-memory measurement performed.")

Record power mode, synchronized GPU timing, warmup, median/p95 latency and
peak allocated/reserved memory in future experiments. CPU execution here does
not establish GPU readiness.

## 20. Conclusions and next experiment

Publish only facts established by this execution. Readiness requires complete
official data, integrity and leakage checks, frozen partitions and an appropriate
environment. This notebook does not produce benchmark model results.

In [ ]:
status = {"dataset_available_and_validated": READY,
    "categories_discovered": list(found), "missing_categories": missing,
    "observed_image_count": int(len(frame)) if READY else None,
    "configuration_sha256": config_hash(config),
    "cuda_available": environment.get("cuda_available", False),
    "test_exposure": "descriptive images/masks only if data available; no tuning",
    "status": "EDA data sections executed" if READY else "BLOCKED: missing complete dataset"}
if READY:
    status["partition_manifest_sha256"] = manifest_sha
    status["counts"] = frame.groupby(["category", "split", "defect"]).size().rename("count").reset_index().to_dict("records")
(OUT / "status.json").write_text(json.dumps(status, indent=2) + "\n")
display(status)

**Next:** acquire and validate missing data when blocked, resolve CUDA readiness,
then begin **Notebook 01 — Basic CNN** with an explicitly declared normal-only
objective and append-only experiment records. Do not implement all models or
the engineering application at this stage.